In [25]:
!pip install tensorflow

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
import librosa
import librosa.display
import IPython.display as pld
import warnings
warnings.filterwarnings('ignore')
import csv
from sklearn.model_selection import train_test_split
import tensorflow as tf

import tensorflow.keras as keras


In [27]:
nums = range(1,190)
print(list(map(str, nums)))

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '

In [28]:
#q = signal.windows.hamming(512)
sr = 22050

dataset_path = r"C:\Users\parth\Documents\COLD_DATASET_(common)"
#dataset_path = 'C:/Users/User/Desktop/test/test1'
#MS_path='C:/Users/User/Desktop/URTIC_MEL_Spectrogtams/test_MS_25'

header = ['filename', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189','190',
          'Label']



with open('develop_with_silence_spectral_25DB_mean.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)

In [31]:
counter=0
for i, (dirpath, dirnames, filenames) in enumerate(os.walk(dataset_path)):
    # ensure we're at sub-folder level
    if dirpath is not dataset_path:
        label = dirpath.split("\\")[-1]

        for f in filenames:
            file_name = f
            file_path = os.path.join(dirpath, f)
            new_file_name=os.path.splitext(file_name)[0]
            
            counter=counter+1
            print(counter)
            
            # load audio file and slice it to ensure length consistency among different files
            signal, sample_rate = librosa.load(file_path)

            signal=librosa.effects.preemphasis(y=signal,  coef=0.95, zi=None, return_zf=False)
            
            # remove silence
            signal_1 = librosa.effects.split(y=signal, top_db=25,  frame_length=512, hop_length=256)
            l = []
            for i in signal_1:
                l.append(signal[i[0]:i[1]])
            signal = np.concatenate(l, axis=0)
            
            mfcc=librosa.feature.mfcc(y=signal, sr=22050,  n_mfcc=13, n_fft=512, hop_length=256)
            mfcc_mean=np.mean(mfcc,  axis=1)
            delta_mfcc = librosa.feature.delta(mfcc)
            delta_mfcc_mean=np.mean(delta_mfcc,  axis=1)
            delta2_mfcc = librosa.feature.delta(mfcc, order=2)
            delta2_mfcc_mean=np.mean(delta2_mfcc,  axis=1)
#             print(mfcc.shape)
#             print(mfcc_mean.shape)
#             print(delta_mfcc.shape)
#             print(delta_mfcc_mean.shape)            
#             print("***mfcc**")
            

            mel_spectro = librosa.feature.melspectrogram(y=signal, sr=sr, n_fft=512, hop_length=256)
            mel_spectro_mean=np.mean(mel_spectro,  axis=1)
#             print(mel_spectro.shape)
#             print(mel_spectro_mean.shape)
#             print("**mel_spectro***")
            
            
            chroma_stft1=librosa.feature.chroma_stft(y=signal, sr=22050,  n_fft=512, hop_length=256,  n_chroma=12)
            chroma_stft1_mean=np.mean(chroma_stft1, axis=1)
#             print(chroma_stft1.shape)
#             print(chroma_stft1_mean.shape)
#             print("**chroma_stft1***")
            
            spectral_contrast1=librosa.feature.spectral_contrast(y=signal, sr=22050,  n_fft=512, hop_length=256,  center=True, pad_mode='constant', freq=None, fmin=200.0, n_bands=6)
            spectral_contrast1_mean=np.mean(spectral_contrast1, axis=1) 
#             print(spectral_contrast1.shape)
#             print(spectral_contrast1_mean.shape)
#             print("***spectral_contrast1**")            
            
            spectral_centroid1=librosa.feature.spectral_centroid(y=signal, sr=22050,  n_fft=512, hop_length=256)
            spectral_centroid1_mean=np.mean(spectral_centroid1, axis=1)
#             print(spectral_centroid1.shape)
#             print(spectral_centroid1_mean.shape)
#             print("**spectral_centroid1***")
            
            spectral_bandwidth1=librosa.feature.spectral_bandwidth(y=signal, sr=22050,   n_fft=512, hop_length=256)
            spectral_bandwidth1_mean=np.mean(spectral_bandwidth1, axis=1)
#             print(spectral_bandwidth1.shape)
#             print(spectral_bandwidth1_mean.shape)
#             print("***spectral_bandwidth1**")            
            
            spectral_flatness1=librosa.feature.spectral_flatness(y=signal,   n_fft=512, hop_length=256)
            spectral_flatness1_mean=np.mean(spectral_flatness1, axis=1)  
#             print(spectral_flatness1.shape)
#             print(spectral_flatness1_mean.shape)
#             print("**spectral_flatness1***")
            
            spectral_rolloff1=librosa.feature.spectral_rolloff(y=signal, sr=22050, n_fft=512, hop_length=256)
            spectral_rolloff1_mean=np.mean(spectral_rolloff1, axis=1)  
#             print(spectral_rolloff1.shape)
#             print(chroma_stft1_mean.shape)
#             print("***spectral_rolloff1**")            
            
            

            final_data=[]
            final_data.append(new_file_name)
            final_data.extend(mfcc_mean)
            final_data.extend(delta_mfcc_mean)
            final_data.extend(delta2_mfcc_mean)
            final_data.extend(mel_spectro_mean)
            final_data.extend(chroma_stft1_mean)  
            final_data.extend(spectral_contrast1_mean)
            final_data.append(spectral_centroid1_mean[0])
            final_data.append(spectral_bandwidth1_mean[0])
            final_data.append(spectral_flatness1_mean[0])
            final_data.append(spectral_rolloff1_mean[0])
            final_data.append(label)



            with open('develop_with_silence_spectral_25DB_mean.csv','a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(final_data)



1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
277
